# Which source-only fitting methods best predict target scores and winners?

The numerical output below was executed from saved study results. To recompute the underlying analysis, run `python tools/empirical_study.py --stage 5` from the repository root, after prerequisite stages. `tools/study_reproduce.py` runs the complete offline sequence into a fresh directory.

# Stage 5 Extrapolation specification

Predictions target 1B, using every source cutoff from the fourth available size (10M) to 750M. All 25 recipes, three-seed means, equal scale weights and identical permitted source ranges are used for every method. The largest-observed baseline uses only the final observation within that permitted range. It is intentionally a ranking baseline; using its unchanged score as a numeric prediction may have substantial level bias.

Let x=N/1,000,000. Linear is a+b*x. Log-linear is a+b*ln(x), not a logarithm of the score. Power-law is E+A*x**(-alpha), alpha in [0.01,3], E and A unconstrained. At each alpha the two linear coefficients are solved by least squares. A fixed 60-point log grid plus bounded refinements of local minima and endpoints determines the minimum. This normalized, profiled implementation was selected before examining target results. It shares the existing PowerLawN functional form but intentionally does not inherit its unit-dependent amplitude bounds. Scores are not clipped. Boundary fits and out-of-range predictions are retained and reported. Alpha bounds are a modeling assumption, examined in Stage 6, not tuned against target scores.

Fitting receives source scores and target size only; target scores enter solely after prediction. Full failures are logged and are not silently dropped. Headline summaries average 10 tasks and 10 cutoffs equally; they are descriptive on a fixed, dependent panel. Method performance at each cutoff is also saved. RMSE and correct winner selection answer different questions. A smaller RMSE does not imply smaller selection regret.


In [1]:
from pathlib import Path
import pandas as pd
ROOT = Path.cwd() if (Path.cwd()/'tools').exists() else Path.cwd().parent
result = pd.read_csv(ROOT/'results/empirical_study/extrapolation_summary.csv')
print(result.to_string(index=False))


          method     rmse      mae  spearman  top1  top5_hit   regret
largest_observed 0.153517 0.150969  0.575303  0.27      0.69 0.020285
          linear 1.396924 1.233606  0.546015  0.23      0.58 0.019030
      log_linear 0.129265 0.121641  0.582685  0.26      0.63 0.018958
       power_law 0.127117 0.120637  0.532669  0.26      0.57 0.018317


In [2]:
import numpy as np
import runpy
predict = runpy.run_path(str(ROOT/'tools/empirical_study.py'))['predict_curve']
obs = pd.read_csv(ROOT/'results/empirical_study/analysis_observations.csv',float_precision='round_trip')
g=obs[(obs.task=='arc_easy')&(obs.recipe=='C4')].groupby('params_num').score.mean()
training=g[g.index<=750_000_000]; reference=g.loc[1_000_000_000]
saved=pd.read_csv(ROOT/'results/empirical_study/extrapolation_predictions.csv')
for method in ['largest_observed','linear','log_linear','power_law']:
    pred,_,_=predict(training.index,training,1_000_000_000,method)
    row=saved[(saved.task=='arc_easy')&(saved.recipe=='C4')&(saved.cutoff==750_000_000)&(saved.method==method)].iloc[0]
    assert np.isclose(row.prediction,pred,atol=1e-10)
    print(method, 'prediction',round(pred,5),'held-out observed',round(reference,5))

largest_observed prediction 0.54938 held-out observed 0.64618
linear prediction 0.70022 held-out observed 0.64618
log_linear prediction 0.56149 held-out observed 0.64618
power_law prediction 0.55992 held-out observed 0.64618


![Study result](../figures/empirical_study/extrapolation_error_comparison.png)

## Interpretation

# Stage 7 Integrated findings

## Three primary findings
1. Across the ten individual tasks, mean Spearman correlation rises from 0.130 at 4M to 0.830 at 530M, then falls to 0.787 at 750M. At those sizes, winner selection succeeds on 2/10, 4/10 and 5/10 tasks. Average regret is 4.60, 0.71 and 0.93 accuracy percentage points. Evidence: scale_summary.csv and cross_scale_predictability.csv.
2. Mean ranking correlation across source sizes is 0.954 for ARC-Easy, 0.060 for BoolQ and 0.040 for Winogrande. The aggregate winner is never exactly selected at a smaller size (0/13), which does not imply all selections are materially worse. Evidence: task_heterogeneity.csv and task_ranking_heatmap.png.
3. Across 100 individual task/cutoff settings, the largest-observed baseline has top-1 success 27%, versus 23% linear and 26% for both log-linear and power-law. Power-law has the lowest average RMSE (0.127) and average regret (1.83 percentage points), but not the strongest ranking correlation or winner rate. At the 750M cutoff, linear has lower RMSE and regret than power-law. Evidence: extrapolation_metrics.csv. These are descriptive comparisons, not proof of superiority.

## Secondary findings
Single- and two-seed subsets match the full-seed winner 46.9% and 67.9% of the time. The nested reference makes this optimistic relative to an independent run. Absolute Welch statistics below 1 versus at least 4 are associated with different cross-scale agreement, but do not identify the causal source of instability. Evidence: seed_subsampling_summary.csv and signal_to_noise_summary.csv.

All 3300 historical calibrated pair classifications reproduce without discrepancy. At alpha=0.05 there are 32 reversing, 1424 stable and 1844 within-noise trajectories. Across alpha 0.01 to 0.10, reversals range from 7 to 72. The count is sensitive, although resolved reversals remain a small fraction of this panel. This is not evidence that the true reversal rate is low.

## Important limitations and contrary evidence
2344/2750 power-law fits reach an exponent bound. 1460 linear predictions fall outside [0,1]. These diagnostics are retained, not hidden; power-law parameters should not be interpreted as established scaling exponents. The fitting range and model assumptions materially affect results. Repeated sizes, shared recipes and correlated tasks are not independent replications. Target averages remain noisy. No new model training, measured compute savings or general algorithm guarantee is established.

## Narrative
Small-scale experiments contain useful but task-dependent ranking information. Better average ranking does not ensure correct winner selection. Seed uncertainty, target measurement noise and extrapolation choices all limit confidence. A defensible conclusion is to evaluate rankings, regret and method diagnostics together, with abstention or additional evidence where decisions are uncertain.

The original DataDecide observations belong to the released dataset authors. This study's contribution is the additional offline analysis and validation of that dataset. Existing numerical debugging evidence is separate from these empirical observations. The report is a course self-study, not a claim of new large-scale training or a new general theorem.


In [ ]:
# Optional regeneration (may take several minutes):
# import subprocess, sys
# subprocess.run([sys.executable, str(ROOT/'tools/empirical_study.py'), '--stage', '5'], cwd=ROOT, check=True)
